# Diagnostics & results

A **control panel** for the three post-run diagnostics in this project:

1. compressor (stage-1) training curves
2. NLE (stage-2) training curves
3. final evaluation (stage 4): SBC, corner plots, metrics table

Each section calls the *same script you'd run on the command line*, so there is no
duplicated logic and no drift. Run this **on `tycho`** (via the VS Code tunnel) so it
can see your scratch data. Edit the **Configuration** cell, then *Run All*.

> To make figures appear in the published Jupyter Book, run this notebook on the
> cluster and **save it with its outputs** before committing (see `notebooks/README.md`).

## Setup
Run everything from the repo root so `tools/…` and `eval.py` resolve, wherever Jupyter was launched.

In [ ]:
import os, glob, subprocess
from pathlib import Path
from IPython.display import Image, display

REPO = subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).decode().strip()
os.chdir(REPO)
OUT = Path('notebooks/_figs'); OUT.mkdir(parents=True, exist_ok=True)
print('working dir:', REPO)

## Configuration — edit these
Point these at the run you want to inspect. `ARM_NLE` is the arm's stage-2 output
directory, i.e. `<scratch_root>/<arm_name>/nle` from that arm's config.

In [ ]:
CONFIG  = 'configs_seeds/noise/arm_cnn_vmim_jitter_n1.yaml'  # the arm's config YAML (in-repo)
ARM_NLE = '/scratch/CHANGE_ME/arm_cnn_vmim_jitter_n1/nle'    # <-- edit: arm's stage-2 output dir
LABEL   = 'VMIM n1'   # legend label
FAMILY  = 'nsf'       # density family trained in stage 2 (gmm | maf | nsf)
SCOPE   = 'std'       # 'std' (standardized t) or 'raw'

## 1. Compressor training
Calls `tools/plot_training_compressor.py` and shows the figure inline.

In [ ]:
fig = OUT / 'compressor.pdf'
!python tools/plot_training_compressor.py "vmim={ARM_NLE}" --names "{LABEL}" --out "{fig}" --png
png = fig.with_suffix('.png')
display(Image(str(png))) if png.exists() else print('No PNG produced — check the ARM_NLE path.')

## 2. NLE training
Calls `tools/plot_training_nle.py` on the config(s).

In [ ]:
fig = OUT / 'nle_training.png'
!python tools/plot_training_nle.py "{CONFIG}" --out "{fig}"
display(Image(str(fig))) if fig.exists() else print('No figure — check CONFIG / stage-2 outputs.')

## 3. Final evaluation (stage 4)
Runs `eval.py`, shows the **metrics table** inline, and previews each generated
PDF figure (SBC, corner). PDF preview uses poppler's `pdftoppm`; if it isn't
installed it simply lists the figure paths instead.

In [ ]:
evdir = OUT / 'eval'
!python eval.py --item "{CONFIG}|{FAMILY}|{SCOPE}|{LABEL}" --out "{evdir}"

# metrics table
try:
    import pandas as pd
    mcsv = evdir / 'metrics.csv'
    if mcsv.exists():
        display(pd.read_csv(mcsv))
except Exception as e:
    print('metrics table skipped:', e)

# preview PDF figures (SBC, corner, ...)
for p in sorted(glob.glob(str(evdir / '*.pdf'))):
    prev = p[:-4] + '_preview'
    rc = os.system(f'pdftoppm -png -r 110 -singlefile "{p}" "{prev}" 2>/dev/null')
    if rc == 0 and os.path.exists(prev + '.png'):
        print(os.path.basename(p)); display(Image(prev + '.png'))
    else:
        print('figure:', p)

---
*Generated figures live under `notebooks/_figs/` (git-ignored). To publish them in
the Jupyter Book, keep this notebook's saved outputs when you commit it.*